# Giai đoạn 4: Đánh giá chéo BDD và audit baseline

Notebook này chỉ phục vụ đánh giá sau train. Không chạy tự động khi mở file. Hãy chạy từng block trong VSCode và kiểm tra output sau mỗi block.

## Mục tiêu
- Đánh giá `bdd_day_base_ep50` trên cả validation day và night.
- Đánh giá `bdd_night_ft_from_bdd_day_ep50` trên cả validation night và day.
- Dùng cùng `imgsz`, `batch`, `workers` và `cache=False` để so sánh công bằng mà không tạo thêm cache ảnh rất lớn.
- Ghi metrics, đường dẫn output và mẫu dự đoán vào thư mục đánh giá.
- Audit các artifact Raw, CLAHE, Gamma, MSRCR, Median Filter, GAN và artifact legacy trước khi đưa vào bảng chính.

## Quy tắc kết luận
- Chỉ dùng kết quả có dataset, checkpoint, metric và provenance xác định được trong bảng chính.
- Artifact thiếu metadata hoặc không chứng minh được cùng input/label sẽ chỉ ghi trong audit, không dùng để kết luận.
- `best.pt` là checkpoint chính; `last.pt` chỉ dùng để đối chiếu hoặc khôi phục khi cần.
- Không dùng notebook này để train hoặc preprocess lại BDD.

In [1]:
import csv
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

import torch
import ultralytics
from ultralytics import YOLO

print(f'Python: {sys.version.split()[0]}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA build: {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    raise RuntimeError('CUDA không khả dụng; dừng trước khi đánh giá.')

Python: 3.10.11
Ultralytics: 8.4.80
PyTorch: 2.5.1+cu121
CUDA build: 12.1
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.29 GB


## Cấu hình và kiểm tra đầu vào

Các đường dẫn dưới đây khớp với repo Windows hiện tại. Nếu repo được chuyển máy, chỉ sửa `BASE_DIR`; không sửa rải rác từng path trong các block sau.

In [2]:
BASE_DIR = Path('d:/DAT301m/proposal')
RUNS_DIR = BASE_DIR / 'models' / 'runs'
EVAL_DIR = RUNS_DIR / 'bdd_evaluation'
DOCS_DIR = BASE_DIR / 'docs'

DATASETS = {
    'day': {
        'yaml': BASE_DIR / 'data' / 'bdd_day.yaml',
        'root': BASE_DIR / 'data' / 'processed' / 'bdd_day',
    },
    'night': {
        'yaml': BASE_DIR / 'data' / 'bdd_night.yaml',
        'root': BASE_DIR / 'data' / 'processed' / 'bdd_night',
    },
}

MODELS = {
    'day_best': RUNS_DIR / 'bdd_day_base_ep50' / 'weights' / 'best.pt',
    'night_best': RUNS_DIR / 'bdd_night_ft_from_bdd_day_ep50' / 'weights' / 'best.pt',
}

for name, item in DATASETS.items():
    if not item['yaml'].exists():
        raise FileNotFoundError(f'Thiếu YAML {name}: {item["yaml"]}')
    if not item['root'].exists():
        raise FileNotFoundError(f'Thiếu dataset {name}: {item["root"]}')
for name, path in MODELS.items():
    if not path.exists():
        raise FileNotFoundError(f'Thiếu checkpoint {name}: {path}')

EVAL_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
print('Đã xác nhận đủ YAML, dataset và checkpoint.')
print(f'Thư mục output đánh giá: {EVAL_DIR}')

Đã xác nhận đủ YAML, dataset và checkpoint.
Thư mục output đánh giá: d:\DAT301m\proposal\models\runs\bdd_evaluation


## Validation chéo day/night

Block này có thể mất nhiều thời gian vì phải chạy bốn lượt validation. Không bật `cache='disk'`: mục tiêu là tránh sinh thêm hàng chục GB `.npy` trong khi validation vẫn đọc được ảnh từ dataset đã preprocess.

In [3]:
EVAL_CASES = [
    ('day_model_on_day', 'day_best', 'day'),
    ('day_model_on_night', 'day_best', 'night'),
    ('night_model_on_night', 'night_best', 'night'),
    ('night_model_on_day', 'night_best', 'day'),
]

def run_validation(case_name, model_key, dataset_key):
    model = YOLO(str(MODELS[model_key]))
    result = model.val(
        data=str(DATASETS[dataset_key]['yaml']),
        split='val',
        imgsz=640,
        batch=8,
        device=0,
        workers=4,
        cache=False,
        project=str(EVAL_DIR),
        name=case_name,
        exist_ok=True,
        plots=True,
        verbose=True,
    )
    box = result.box
    return {
        'case': case_name,
        'model': model_key,
        'dataset': dataset_key,
        'precision': float(box.mp),
        'recall': float(box.mr),
        'mAP50': float(box.map50),
        'mAP50-95': float(box.map),
        'output_dir': str(EVAL_DIR / case_name),
    }

validation_rows = []
for case_name, model_key, dataset_key in EVAL_CASES:
    print(f'\nĐang đánh giá: {case_name}')
    validation_rows.append(run_validation(case_name, model_key, dataset_key))

metrics_path = EVAL_DIR / 'bdd_cross_domain_metrics.csv'
with metrics_path.open('w', newline='', encoding='utf-8-sig') as handle:
    writer = csv.DictWriter(handle, fieldnames=validation_rows[0].keys())
    writer.writeheader()
    writer.writerows(validation_rows)

print(f'Đã ghi metrics: {metrics_path}')


Đang đánh giá: day_model_on_day
Ultralytics 8.4.80  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11n summary (fused): 101 layers, 2,584,102 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 94.621.7 MB/s, size: 63.1 KB)
val: Scanning D:\DAT301m\proposal\data\processed\bdd_day\val\labels... 5258 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5258/5258 1.5Kit/s 3.6s0.1s
val: New cache created: D:\DAT301m\proposal\data\processed\bdd_day\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 658/658 16.6it/s 39.6s0.1ss
                   all       5258     105388      0.674      0.399      0.442      0.248
            pedestrian       2133       9366      0.648      0.482      0.533      0.253
                 rider        369        478       0.59      0.299      0.341      0.163
                   car       5206      57833      0.743  

## Mẫu inference trực quan

Chạy sau validation. Mẫu này chỉ lấy tối đa 12 ảnh validation mỗi miền để kiểm tra trực quan, không dùng thay cho mAP.

In [4]:
def sample_images(dataset_key, limit=12):
    image_dir = DATASETS[dataset_key]['root'] / 'val' / 'images'
    return sorted(
        [p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    )[:limit]

for dataset_key in ('day', 'night'):
    images = sample_images(dataset_key)
    for model_key in ('day_best', 'night_best'):
        case_name = f'{model_key}_samples_on_{dataset_key}'
        model = YOLO(str(MODELS[model_key]))
        model.predict(
            source=[str(p) for p in images],
            imgsz=640,
            device=0,
            conf=0.25,
            save=True,
            project=str(EVAL_DIR),
            name=case_name,
            exist_ok=True,
            verbose=False,
        )
        print(f'Đã ghi mẫu: {EVAL_DIR / case_name}')

Results saved to D:\DAT301m\proposal\models\runs\bdd_evaluation\day_best_samples_on_day
Đã ghi mẫu: d:\DAT301m\proposal\models\runs\bdd_evaluation\day_best_samples_on_day
Results saved to D:\DAT301m\proposal\models\runs\bdd_evaluation\night_best_samples_on_day
Đã ghi mẫu: d:\DAT301m\proposal\models\runs\bdd_evaluation\night_best_samples_on_day
Results saved to D:\DAT301m\proposal\models\runs\bdd_evaluation\day_best_samples_on_night
Đã ghi mẫu: d:\DAT301m\proposal\models\runs\bdd_evaluation\day_best_samples_on_night
Results saved to D:\DAT301m\proposal\models\runs\bdd_evaluation\night_best_samples_on_night
Đã ghi mẫu: d:\DAT301m\proposal\models\runs\bdd_evaluation\night_best_samples_on_night


## Audit artifact enhancement legacy

Danh sách dưới đây chỉ là ứng viên cần kiểm tra. Notebook không tự coi artifact là baseline hợp lệ. Trước khi đưa vào bảng chính, cần xác nhận cùng tập ảnh, cùng nhãn, cùng checkpoint/protocol và có metadata đủ rõ. Artifact không đạt điều kiện phải đánh dấu `eligible_for_main_table=False`.

In [5]:
ARTIFACT_CANDIDATES = {
    'clahe': BASE_DIR / 'baseline_comparison_eval15' / 'enhanced' / 'clahe',
    'gamma': BASE_DIR / 'baseline_comparison_eval15' / 'enhanced' / 'gamma',
    'msrcr': BASE_DIR / 'baseline_comparison_eval15' / 'enhanced' / 'msrcr',
    'median_filter': BASE_DIR / 'baseline_comparison_eval15' / 'enhanced' / 'low_light',
    'gan_legacy': BASE_DIR / 'baseline_comparison_eval15' / 'enhanced' / 'gan',
    'combined_eval': BASE_DIR / 'new' / 'baseline_comparison_combined_eval',
    'tong_hop_eval': BASE_DIR / 'tong_hop' / 'tong_hop' / 'evaluation_eval15',
}

audit_rows = []
for name, path in ARTIFACT_CANDIDATES.items():
    files = list(path.rglob('*')) if path.exists() else []
    files = [p for p in files if p.is_file()]
    total_bytes = sum(p.stat().st_size for p in files)
    audit_rows.append({
        'artifact': name,
        'path': str(path),
        'exists': path.exists(),
        'file_count': len(files),
        'size_gb': round(total_bytes / (1024 ** 3), 3),
        'eligible_for_main_table': False,
        'provenance_status': 'manual_review_required',
        'notes': 'Chỉ đưa vào bảng chính sau khi xác nhận input, label, protocol và metadata.',
    })

audit_path = DOCS_DIR / 'enhancement_artifact_audit.csv'
with audit_path.open('w', newline='', encoding='utf-8-sig') as handle:
    writer = csv.DictWriter(handle, fieldnames=audit_rows[0].keys())
    writer.writeheader()
    writer.writerows(audit_rows)

print(f'Đã ghi audit artifact: {audit_path}')
for row in audit_rows:
    print(row)

Đã ghi audit artifact: d:\DAT301m\proposal\docs\enhancement_artifact_audit.csv
{'artifact': 'clahe', 'path': 'd:\\DAT301m\\proposal\\baseline_comparison_eval15\\enhanced\\clahe', 'exists': True, 'file_count': 15, 'size_gb': 0.001, 'eligible_for_main_table': False, 'provenance_status': 'manual_review_required', 'notes': 'Chỉ đưa vào bảng chính sau khi xác nhận input, label, protocol và metadata.'}
{'artifact': 'gamma', 'path': 'd:\\DAT301m\\proposal\\baseline_comparison_eval15\\enhanced\\gamma', 'exists': True, 'file_count': 15, 'size_gb': 0.001, 'eligible_for_main_table': False, 'provenance_status': 'manual_review_required', 'notes': 'Chỉ đưa vào bảng chính sau khi xác nhận input, label, protocol và metadata.'}
{'artifact': 'msrcr', 'path': 'd:\\DAT301m\\proposal\\baseline_comparison_eval15\\enhanced\\msrcr', 'exists': True, 'file_count': 15, 'size_gb': 0.002, 'eligible_for_main_table': False, 'provenance_status': 'manual_review_required', 'notes': 'Chỉ đưa vào bảng chính sau khi xác n

## Checklist sau khi chạy

- Kiểm tra `models/runs/bdd_evaluation/bdd_cross_domain_metrics.csv`.
- Kiểm tra bốn thư mục validation và mẫu inference.
- Kiểm tra `docs/enhancement_artifact_audit.csv`; chỉ chuyển artifact đủ provenance vào bảng chính.
- Cập nhật `task.md` bằng số liệu thực tế sau khi chạy xong.
- Chỉ sau khi các output cần thiết đã được sao lưu mới dọn `labels.cache` và `*.npy`.
- Không xóa raw data, YAML, manifest, `best.pt`, `last.pt` hoặc weight gắn nhãn.